**Source: https://gym.openai.com/envs/CartPole-v1/
<br>Import environment here**

**Video illustration of the 5 episodes/games look on youtube - https://youtu.be/7knoFtadY_0**

In [1]:
import pandas as pd
import numpy as np
import random
import math
import gym

In [2]:
env = gym.make('CartPole-v1')

In [3]:
 """
    Description:
        A pole is attached by an un-actuated joint to a cart, which moves along
        a frictionless track. The pendulum starts upright, and the goal is to
        prevent it from falling over by increasing and reducing the cart's
        velocity.
    Source:
        This environment corresponds to the version of the cart-pole problem
        described by Barto, Sutton, and Anderson
    Observation:
        Type: Box(4)
        Num     Observation               Min                     Max
        0       Cart Position             -4.8                    4.8
        1       Cart Velocity             -Inf                    Inf
        2       Pole Angle                -0.418 rad (-24 deg)    0.418 rad (24 deg)
        3       Pole Angular Velocity     -Inf                    Inf
    Actions:
        Type: Discrete(2)
        Num   Action
        0     Push cart to the left
        1     Push cart to the right
        Note: The amount the velocity that is reduced or increased is not
        fixed; it depends on the angle the pole is pointing. This is because
        the center of gravity of the pole increases the amount of energy needed
        to move the cart underneath it
    Reward:
        Reward is 1 for every step taken, including the termination step. 
        Episod end if angle is greater than 15 deg or 0.2617993878 rad
    Starting State:
        All observations are assigned a uniform random value in [-0.05..0.05]
    Episode Termination:
        Pole Angle is more than 12 degrees.
        Cart Position is more than 2.4 (center of the cart reaches the edge of
        the display).
        Episode length is greater than 200.
        Solved Requirements:
        Considered solved when the average return is greater than or equal to
        195.0 over 100 consecutive trials.
    """


"\n   Description:\n       A pole is attached by an un-actuated joint to a cart, which moves along\n       a frictionless track. The pendulum starts upright, and the goal is to\n       prevent it from falling over by increasing and reducing the cart's\n       velocity.\n   Source:\n       This environment corresponds to the version of the cart-pole problem\n       described by Barto, Sutton, and Anderson\n   Observation:\n       Type: Box(4)\n       Num     Observation               Min                     Max\n       0       Cart Position             -4.8                    4.8\n       1       Cart Velocity             -Inf                    Inf\n       2       Pole Angle                -0.418 rad (-24 deg)    0.418 rad (24 deg)\n       3       Pole Angular Velocity     -Inf                    Inf\n   Actions:\n       Type: Discrete(2)\n       Num   Action\n       0     Push cart to the left\n       1     Push cart to the right\n       Note: The amount the velocity that is reduced or

In [4]:
env.action_space
# 0 - go left
# 1 - go right

Discrete(2)

In [5]:
env.reset() 
# 0 - car position [-4.8, +4.8]
# 1 - car velocity [-inf, +inf]
# 2 - pole angle [-0.418, + 0.418] rad or [-24, +24] deg
# 3 - pole angle velocity [-inf, +inf]

array([ 0.01736495, -0.02325174, -0.02473375, -0.03310925])

In [6]:
# Linear model is deterministic since transition probability = 1.0
# V(s) = FAI_T * PSI(S), where PSI(s) = (x, x', Fi, Fi') - vector of state
# Apply Fitted value iteration algorithm
# PI_star(s) = argmin(V*(s,a))

In [12]:
def model(state, action):
        gravity = 9.8
        masscart = 1.0
        masspole = 0.1
        total_mass = masspole + masscart
        length = 0.5  # actually half the pole's length
        polemass_length = masspole * length
        force_mag = 10.0
        tau = 0.02  # seconds between state updates

        x, x_dot, theta, theta_dot = state
        if action == 1:
            force = force_mag
        else:
            force = -force_mag

        costheta = math.cos(theta)
        sintheta = math.sin(theta)

        # For the interested reader:
        # https://coneural.org/florian/papers/05_cart_pole.pdf
        temp = (
            force + polemass_length * theta_dot ** 2 * sintheta
        ) / total_mass
        thetaacc = (gravity * sintheta - costheta * temp) / (
            length * (4.0 / 3.0 - masspole * costheta ** 2 / total_mass)
        )
        xacc = temp - polemass_length * thetaacc * costheta / total_mass


        x = x + tau * x_dot
        x_dot = x_dot + tau * xacc
        theta = theta + tau * theta_dot
        theta_dot = theta_dot + tau * thetaacc

        res = np.array([x, x_dot, theta, theta_dot])
        
        return res

In [9]:
#FVI - without acceleration
m = 1000 # number of initial states s1,...sm
d = 0.99 # discount factor
alp = 0.001 # step of descent 
diff = 1.0 
eps = 0.0001
num = 0 # iteration number

s_arr = np.array([np.zeros(4) * i for i in range(m)]) #array for sample of m random states
FAI = np.zeros(4) #initialize vector of param = 0
FAI_old = np.zeros(4)
arr = np.zeros(2) # array q(a)
y = np.zeros(m) #array for min[R(s,a) + d * E[V(s')]]

for i in range(m): #make m random sample of states
    place = env.reset()
    for j in range(4):
        s_arr[i][j] = place[j]
        
env.reset() #reset the enviroment 

while diff > eps:
    for i in range(m): #for each state of array s_arr
        for j in range(2): #for each action 0 or 1
            
            s_new = model(s_arr[i], j) #new state
        
            if abs(s_new[2]) <= 0.2617993878: #if angle is less than 15
                r = 1.0
            else:
                r = 0.0
                print(i, "GG")
            
            val = d * np.sum(FAI_old * s_new) #discount_fact * E[V[s']]
            arr[j] = r + val #q(a) = R(s_new) + discount_fact * E[V[s']]
        y[i] = np.amin(arr) #y_i = min q(a) wrt a 
        
    for j in range(4):
        der = 0.0 #initial derivative value
        p = random.randint(1,2)
        if j == 1:
            p = 1
        elif j == 3:
            p = 2
            
        for l in range(m):
            der += (np.sum(FAI_old * abs(s_arr[l])) - y[l]) * abs(s_arr[l][j]) * (-1) ** p
        FAI[j] = FAI_old[j] -  alp * der #descent step
        
    diff = np.sum(abs(FAI- FAI_old))
    num += 1
    
    #delta
    delta = 0.0
    for j in range(m):
        delta += abs(np.sum(FAI * s_arr[j]) - y[j])
    delta *= 1/(m)
        

    for j in range(4):
        FAI_old[j] = FAI[j]
        
    if num == 300:
        print('\n', "Algorithm is finished, Fai is ", FAI)
        m = np.amax(abs(FAI)) #normalize vector FAI
        FAI /= m
        break
        
    #print(diff, delta, FAI, num)


 Algorithm is finished, Fai is  [-0.06001568 -2.12684712 -0.28388787  1.97282953]


In [10]:
def VI_star(state, FAI): #function V*
    res = np.sum(FAI * abs(state))
    return res

In [11]:
#Optimal game - 100 GAMES WITHOUT ILLUSTATION,OUPUT = AVERAGE NUMBER OF STEPS FOR 100 EPISODS
d = {}
arr = np.zeros(2) #array of values(s_new(a = 0) and s_new(a = 1))
R = np.array([])

for i in range(100): #number of the episods/games
    place = env.reset()
    status = False
    r = 0.0
    #env.render()
    
    while (status == False):
        
        for j in range(2):      
            s_new = model(place, j) #obtain new state by taking action j at state = place
            arr[j] = VI_star(s_new, FAI) #value VI_star at s_new
            #print('Value of V*', arr[j], ' ', 'by action', j, ' ', 's_new', s_new)
            d[VI_star(s_new, FAI)] = j
            
        Min = np.amin(arr) #get min value
        action = d[Min] #choose optimal action
        state = env.step(action) #make optimal step
        
        #env.render()
        #print(state,"" ,'\n')
        r += state[1] #reward of making optimal step
        place = state[0] #new place
        status = state[2] #check for end of the episod
        
        if status == True:
            R = np.append(R, r)
            #env.close()  
print('Average number of steps is', np.mean(R))

Average number of steps is 500.0


**Video Illustration of the 5 episodes/games look on youtube - https://youtu.be/7knoFtadY_0**